# Axial Progenitors and tailbud elongation

## Purpose 

To explore the dynamics of tailbud elongation and the quantify the potential of axial progenitors.

## Setup

In [ ]:
import cassiopeia as cas
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycea as py
import scanpy as sc
import seaborn as sns
from devmap.config import (
    axial_palette,
    celltype_palette,
    colors,
    embryo_palette,
    get_paths,
    lca_cmap,
    set_theme,
    stage_palette,
)
from devmap.plots import plot_area
from devmap.progenitors import bipotency_permutation_test, get_fate_progenitors, mark_sibling_descendants
from devmap.utils import load_data, save_plot

set_theme()
base_path, plots_path, results_path = get_paths("vignettes")

## Load data

In [2]:
tdata = load_data("log1p",scvi = True)
spatial_scores = pd.read_csv(results_path / "spatial_scores.csv", index_col=0)
tdata.obs["ap_score"] = spatial_scores["ap_score"]
tdata.obs["dv_score"] = spatial_scores["dv_score"]

## Identify fate-restricted clades

Neural

In [ ]:
tdata.obs["is_neural"] = tdata.obs["lineage"].isin(["Neural ectoderm", "Neural crest"]).astype(int)
py.tl.ancestral_states(tdata, "is_neural", method = "mean")
tdata.obs["is_neural"] = tdata.obs["is_neural"].astype(bool)
neural_progenitors = get_fate_progenitors(tdata, "is_neural", key_added="neural_progenitor", min_descendants=2)

Axial progenitor

In [ ]:
tdata.obs["is_nmp"] = tdata.obs["cell_type"].isin(["Axial progenitor"]).astype(int)
py.tl.ancestral_states(tdata, "is_nmp", method = "mean")
tdata.obs["is_nmp"] = tdata.obs["is_nmp"].astype(bool)
nmp_progenitors = get_fate_progenitors(tdata, "is_nmp", key_added="nmp_progenitor", min_descendants=2)

Mark sibling descendants

In [ ]:
mark_sibling_descendants(tdata, neural_progenitors)
edges = np.linspace(0, .85, 18)
neural_progenitors["ap_bin"] = pd.cut(neural_progenitors["ap_score"], edges,
    labels=edges[:-1].round(2))
tdata.obs["sibling_ap"] = tdata.obs.sibling_descendants.map(neural_progenitors["ap_bin"])
mark_sibling_descendants(tdata, nmp_progenitors)

## Sibling representation along AP axis

In [ ]:
axial_types = {
    "Axial progenitor": "Axial progenitor",
    "Presomitic mesoderm": "Presomitic mesoderm",
    "Nascent somite": "Presomitic mesoderm",
    "Sclerotome": "Somite",
    "Dermomyotome": "Somite",
    "Myotome": "Somite",
    "Hypaxial myogenic progenitor": "Somite",
    "Spinal cord (cervical)": "Spinal cord",
    "Spinal cord (thoracic)": "Spinal cord",
    "Spinal cord (lumbar)": "Spinal cord",
    "Spinal cord (sacral)": "Spinal cord",
    "Hindbrain (r2-r4)": "Hindbrain",
    "Hindbrain (r5-r8)": "Hindbrain",
}
type_order = ['Primordial germ cell', 'Ectoderm', 'Neural ectoderm','Spinal cord', 'Hindbrain',
              'Surface ectoderm', 'Neural crest', 'Extraembryonic ectoderm','Extraembryonic mesoderm',
              'Mesoderm', 'Lateral plate mesoderm', 'Intermediate mesoderm','Paraxial mesoderm', 'Somite',
              "Presomitic mesoderm",'Endoderm','Extraembryonic endoderm', 'Blood','Axial progenitor']
tdata.obs["axial_type"] = tdata.obs["cell_type"].map(axial_types).fillna(tdata.obs["lineage"])

In [ ]:
fig, ax = plt.subplots(figsize=(1.8,1.5), dpi=600)
df = tdata.obs.query("progenitor.isna() & ~is_neural & stage == 'E9.5'").groupby(
    ["axial_type", "sibling_ap"], observed=True).size().reset_index(name="n")
df.axial_type = pd.Categorical(df.axial_type, categories = type_order, ordered=True)
df["color"] = df["axial_type"].map(axial_palette)
plot_area(df, x = "sibling_ap", value = "n", color = "color", ax = ax)
plt.xlim(.1, .8);
plt.xticks([0.2, 0.4, 0.6, 0.8])
plt.ylabel("Sibling output (%)")
save_plot(plots_path / "non_neural_sibling_ap_area_plot.svg", fig, rasterize=True)

## Mesoderm MRCA by AP position

In [ ]:
py.tl.ancestral_linkage(tdata, groupby="germ_layer", target = "Mesoderm", normalize = False,
                        key_added = "mesoderm_mrca", depth_key = "time")

Mean by replicate

In [ ]:
fig, ax = plt.subplots(figsize=(3, 1), dpi=600)

df = tdata.obs.query("stage == 'E9.5' & lineage == 'Neural ectoderm'").copy()
n_bins = 10
z = 1.96  # ~95% CI
norm =  mcolors.Normalize(vmin=7, vmax=9)
cmap = plt.get_cmap("RdPu")

for embryo in ["E9.5-R1", "E9.5-R2", "E9.5-R3"]:
    d = (
        df.query("stage == 'E9.5' & embryo == @embryo")
        .sample(10000, random_state=0)
        .copy()
    )
    d["ap_bin"] = pd.cut(d["ap_score"], bins=n_bins)
    binned = (
        d.groupby("ap_bin", observed=True)
        .agg(
            ap_mean=("ap_score", "mean"),
            y_mean=("Mesoderm_linkage", "mean"),
            y_std=("Mesoderm_linkage", "std"),
            n=("Mesoderm_linkage", "size"),
        )
        .dropna()
        .sort_values("ap_mean")
    )
    ci = z * binned["y_std"] / np.sqrt(binned["n"])
    ax.plot(binned["ap_mean"],binned["y_mean"],color=embryo_palette[embryo],lw=1)

    ax.fill_between(binned["ap_mean"],binned["y_mean"] - ci,binned["y_mean"] + ci,color=embryo_palette[embryo],
        alpha=0.2,linewidth=0)
for y in [7,7.5,8,8.5]:
    ax.axhline(y, color=cmap(norm(y)), linestyle="--", linewidth=0.5, alpha=1, zorder=-1)
ax.set_xlim(.1, .8)
save_plot(plots_path / "neural_mesoderm_mrca_lineplot.svg", fig)

By LCA depth

In [ ]:
bipotent_frac =  []
for timpoint in np.linspace(7, 9, 11):
    df = tdata.obs.query("lineage == 'Neural ectoderm'") \
    .groupby(["ap_bin","stage"], observed=True)["mesoderm_mrca"] \
    .apply(lambda x: (x > timpoint).mean()).reset_index(name="mesoderm_mrca")
    bipotent_frac.append(df.assign(lca = timpoint))
bipotent_frac = pd.concat(bipotent_frac)

fig, ax = plt.subplots(figsize=(1.6,1.6), dpi=600)
bipotent_frac["mesoderm_pct"] = bipotent_frac["mesoderm_mrca"] * 100
sns.lineplot(data=bipotent_frac.query("stage == 'E9.5'"),
             x="ap_bin", y="mesoderm_pct", hue="lca", palette = "RdPu", hue_norm=(6.5,9), linewidth=1, ax = ax)
plt.ylabel("Common ancestor with\nmesoderm (%)")
plt.xlabel("Neural A-P position")
plt.xticks([0,0.4,0.8])
save_plot(plots_path / "bipotent_fraction_by_ap.svg", fig, rasterize=False)

## Posterior cell UMAP

In [507]:
posterior_tdata = tdata[tdata.obs.query("sibling_ap > .6 & stage == 'E9.5'").index]
sc.pp.neighbors(posterior_tdata, use_rep="X_scvi", n_neighbors=50)
sc.tl.umap(posterior_tdata)

colored by axial type

In [2098]:
fig, ax = plt.subplots(figsize=(4,4), dpi=600)
sc.pl.umap(posterior_tdata, color = "axial_type", palette = axial_palette, frameon=False, show = False, ax = ax)
save_plot(plots_path / "nmp_progenitor_axial_umap.svg", fig, rasterize=True)

colored by cell type

In [1148]:
fig, ax = plt.subplots(figsize=(4,4), dpi=600)
show_types = posterior_tdata.obs["cell_type"].value_counts().index[:15].tolist()
sc.pl.umap(posterior_tdata, color = "cell_type", palette = celltype_palette, frameon=False, show = False, ax = ax, groups = show_types)
save_plot(plots_path / "nmp_progenitor_cell_type_umap.svg", fig, rasterize=True)

colored by marker genes

In [1139]:
for gene in ["Sox2", "T", "Meox1"]:
    fig, ax = plt.subplots(figsize=(4,4), dpi=600)
    sc.pl.umap(posterior_tdata, color = gene, vmax = 2.5, frameon=False, show = False, ax = ax)
    save_plot(plots_path / f"nmp_progenitor_{gene}_umap.svg", fig, rasterize=True)

colored by clade

In [1290]:
fig, ax = plt.subplots(figsize=(4,4), dpi=600)
example_clades = posterior_tdata.obs.query("~is_neural").sibling_descendants.value_counts().sample(15).index.to_list()
sc.pl.umap(posterior_tdata, ax = ax, title = "", frameon=False, show = False)
sc.pl.umap(posterior_tdata[posterior_tdata.obs.sibling_descendants.isin(example_clades)], s = 100,
    color = "sibling_descendants", groups = example_clades, ax = ax, title = "", frameon=False, palette = colors)
save_plot(plots_path / "nmp_progenitor_umap.svg", fig, rasterize=True)

colored by clade LCA

In [2496]:
fig, ax = plt.subplots(figsize=(4.2,4), dpi=600)
sc.pl.umap(posterior_tdata, ax = ax, title = "", frameon=False, show = False)
posterior_tdata.obs["progenitor_time"] = posterior_tdata.obs["sibling_descendants"].map(neural_progenitors["time"]).astype(float)
sc.pl.umap(posterior_tdata[posterior_tdata.obs.sibling_descendants.isin(example_clades)], s = 100,vmin = 7, vmax = 9,
    color = "progenitor_time", groups = example_clades, ax = ax, title = "", frameon=False, palette = colors, cmap = "RdPu")
save_plot(plots_path / "nmp_progenitor_depth_umap.svg", fig, rasterize=True)

## AP Position Stripplot

In [ ]:
use_lineages = ["Paraxial mesoderm","Epiblast","Neural crest","Neural ectoderm"]
use_genes = ["Hoxc6", "Hoxc8", "Hoxc9", "Hoxc10", "Hoxc11", "Hoxc12", "Hoxc13", "T", "Sox2", "Meox1", "Mesp2","Hoxc10"]
axial_tdata = tdata[tdata.obs.lineage.isin(use_lineages) & (tdata.obs.stage == "E9.5"), use_genes].copy()
expr = sc.get.obs_df(axial_tdata, keys = use_genes)
for gene in use_genes:
    axial_tdata.obs[f"{gene}_expr"] = expr[gene]
axial_tdata.obs["axis"] = np.clip(0.85 - axial_tdata.obs["ap_score"], -.02, 1)
axial_tdata.obs.loc[axial_tdata.obs["germ_layer"] == "Mesoderm", "axis"] = -axial_tdata.obs.loc[axial_tdata.obs["germ_layer"] == "Mesoderm", "axis"]
axial_tdata.obs.loc[axial_tdata.obs["cell_type"] == "Axial progenitor", "axis"] = axial_tdata.obs.loc[axial_tdata.obs["cell_type"] == "Axial progenitor", "axis"] - 0.05

Define axis

In [2072]:
axial_tdata.obs["axis"] = np.clip(0.80 - axial_tdata.obs["ap_score"], -0.01, 1)
axial_tdata.obs.loc[axial_tdata.obs["germ_layer"] == "Mesoderm", "axis"] = -axial_tdata.obs.loc[axial_tdata.obs["germ_layer"] == "Mesoderm", "axis"]
axial_tdata.obs.loc[axial_tdata.obs["cell_type"] == "Axial progenitor", "axis"] = .8 -axial_tdata.obs.loc[axial_tdata.obs["cell_type"] == "Axial progenitor", "ap_score"]

Plot posterior cell types with AP score

In [2076]:
plot_cell_types = ["Axial progenitor", "Presomitic mesoderm", "Nascent somite", "Sclerotome", "Dermomyotome", "Spinal cord (cervical)", "Spinal cord (thoracic)", "Spinal cord (lumbar)", "Spinal cord (sacral)","Hindbrain (r5-r8)","Hindbrain (r2-r4)"]
fig, ax = plt.subplots(figsize=(3.5,2), dpi=600)
sns.stripplot(
    data=axial_tdata.obs.query("cell_type in @plot_cell_types").sample(30000, random_state=0),
    x="axis",
    jitter=0.2,
    hue = "axial_type", palette = axial_palette, legend=True, size=1,
)
plt.xlim(-.5, .5)
sns.despine(left=True, bottom=True)
plt.yticks([])
plt.xticks([])
plt.yticks([])
# remove axis labels
plt.xlabel("")
plt.ylabel("")
save_plot(plots_path / "axial_genes_stripplot.svg", fig, rasterize=True, transparent=True)

## AP Clade Semicircles

In [ ]:
axial_tdata.obs["progenitor_time"] = axial_tdata.obs["sibling_descendants"].map(neural_progenitors["time"]).astype(float)
axial_tdata.obs["progenitor_time_bin"] = pd.cut(axial_tdata.obs["progenitor_time"], bins=np.linspace(7, 9, 11), labels=np.linspace(7, 9, 11).round(1)[:-1])
df = axial_tdata.obs.query("axial_type.isin(['Spinal cord', 'Somite','Presomitic mesoderm','Hindbrain'])").groupby(["progenitor_time_bin", "is_neural"], observed=True)["axis"].mean().unstack().rename(columns={True:"Ectoderm", False:"Mesoderm"}).reset_index()
fig, ax = plt.subplots(figsize=(3.5, 2), dpi=600)

norm = mcolors.Normalize(vmin=7, vmax=9)
for _, row in df.iterrows():
    x1, x2 = row['Ectoderm'], row['Mesoderm']
    center = (x1 + x2) / 2
    radius = abs(x2 - x1) / 2
    theta = np.linspace(0, np.pi, 100)
    x = center + radius * np.cos(theta)
    y = radius * np.sin(theta)
    ax.plot(x, y, linewidth=1.5,
            color=lca_cmap(norm(row['progenitor_time_bin'])))

ax.set_aspect('equal')   # keep semicircles from looking squashed
ax.set_yticks([])
plt.xlim(-0.5, 0.5)
plt.ylim(0., 0.45)
plt.show()
save_plot(plots_path / "axial_genes_semi_circle.svg", fig)

## AP Smoothed expression

In [ ]:
genes = ["T", "Sox2", "Hoxc9","Meox1","Hoxc6","Hoxc10"]
df = axial_tdata.obs.sort_values("axis").copy()
# choose number of bins or bin width
n_bins = 200
df["axis_bin"] = pd.cut(df["axis"], bins=n_bins)
# bin center for plotting
df["axis_bin_center"] = df["axis_bin"].apply(lambda x: x.mid)
# aggregate mean expression per bin
agg_dict = {f"{g}_expr": "mean" for g in genes}
binned = (
    df.groupby("axis_bin_center", observed=True)
      .agg(agg_dict)
      .reset_index()
      .rename(columns={"axis_bin_center": "axis"})
)
# smooth the binned means
window = 5
for g in genes:
    binned[f"{g}_smooth"] = (
        binned[f"{g}_expr"]
        .rolling(window=window, center=True, min_periods=1)
        .mean()
    )
# reshape for plotting
plot_df = binned.melt(
    id_vars="axis",
    value_vars=[f"{g}_smooth" for g in genes],
    var_name="gene",
    value_name="expression"
)
plot_df["axis"] = plot_df["axis"].astype(float)
plot_df["gene"] = plot_df["gene"].str.replace("_smooth", "", regex=False)
plot_df = plot_df.query("-.5 <= axis <= .5").copy()

fig, ax = plt.subplots(figsize=(3.5,1), dpi=600)
sns.lineplot(data=plot_df, x="axis", y="expression", hue="gene", ax = ax, linewidth=1)
plt.show()
save_plot(plots_path / "axial_genes_lineplot.svg", fig, rasterize=True)

## AP offset

In [ ]:
fig, ax = plt.subplots(figsize=(1.6, 1.6), dpi=600)
df = axial_tdata.obs.query("axial_type.isin(['Spinal cord', 'Somite','Presomitic mesoderm','Hindbrain'])").groupby(["progenitor_time_bin", "is_neural"], observed=True)["ap_score"].mean().unstack().rename(columns={True:"Ectoderm", False:"Mesoderm"}).reset_index()
sns.regplot(data=df, x="Ectoderm", y="Mesoderm", scatter=False, ax=ax, color="black", line_kws={"linewidth": 1})
sns.scatterplot(data=df, x="Ectoderm", y="Mesoderm", hue="progenitor_time_bin", palette="RdPu", ax = ax, legend=False,
                    s=20, edgecolor="black", linewidth=0.5, zorder=3)
# add dashed line y=x
ax.plot([0.45, 0.9], [0.45, 0.8], linestyle="--", color="black", linewidth=1)
# add regression line
mean_offset = (df["Mesoderm"] - df["Ectoderm"]).mean()
# add text with mean offset
ax.text(0.35, 0.75, f"Mean offset: {mean_offset:.2f}", fontsize=8)
#plt.xticks([0.4, 0.6, 0.8])
# r2 
r2 = np.corrcoef(df["Ectoderm"], df["Mesoderm"])[0, 1] ** 2
ax.text(0.35, 0.7, f"R²: {r2:.2f}", fontsize=8)
plt.xlabel("AP score (Ectoderm)")
plt.ylabel("AP score (Mesoderm)")
save_plot(plots_path / "ap_score_scatter.svg", fig, rasterize=False)

## Fraction of bipotent clades containing an axial progenitor

In [ ]:
py.tl.ancestral_linkage(tdata, groupby="cell_type", target = "Axial progenitor", normalize = False,
                        key_added = "axial_mrca", depth_key = "time")
df = tdata.obs.query("stage == 'E9.5' & lineage == 'Neural ectoderm'").copy()
df["contains_axial"] = df["axial_mrca"] >= df["mesoderm_mrca"]
df["bipotent"] = df["mesoderm_mrca"] > 8
nmp_frac = (
    df.query("bipotent == True")
      .groupby("ap_bin")["contains_axial"]
      .mean()
      .reset_index(name="nmp_frac")
)
nmp_frac["nmp_pct"] = nmp_frac["nmp_frac"] * 100

fig, ax = plt.subplots(figsize=(1.6, 1.6), dpi=600)
ax = sns.lineplot(
    data=nmp_frac,
    x="ap_bin",
    y="nmp_pct",
    color="black"
)

# fill area under the curve
ax.fill_between(
    nmp_frac["ap_bin"],
    nmp_frac["nmp_pct"],
    color=celltype_palette["Axial progenitor"],
    alpha=1
)
plt.ylim(0,60)
plt.ylabel("Axial progenitor\nLCA > E8.0 (%)")
plt.xlabel("A-P position")
save_plot(plots_path / "nmp_fraction_by_ap_position.svg", fig, rasterize=True)

## Position of axial progenitors over time

In [ ]:
fig, ax = plt.subplots(figsize=(1.6,1.6), dpi=600)
embryo_somite_counts = {"E8.0-R2": 5, "E8.0-R3": 5, "E8.5-R1": 7, "E8.5-R2": 7
    , "E8.5-R3": 10, "E9.0-R1":14, "E9.0-R2": 16, "E9.0-R3": 13, "E9.5-R1": 27.5, "E9.5-R2": 25.5, "E9.5-R3": 27, "E10.0-R1": 30}
df = tdata.obs.query("cell_type == 'Axial progenitor'").groupby(["embryo","stage"], observed = True)["ap_score"].mean().reset_index()
df["somite_count"] = df.embryo.map(embryo_somite_counts)
df["embryo"] = df.embryo.astype("str")
df = df.query("somite_count.notna()").copy()
sns.regplot(data=df, x="somite_count", y="ap_score", scatter=False, color="gray", line_kws={"linewidth": 1}, ax = ax)
sns.scatterplot(data=df, x="somite_count", y="ap_score", palette=stage_palette, hue="stage", edgecolor="black", linewidth=0.5, ax = ax, s = 30,zorder=2)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel("Somite count")
plt.ylabel("Mean Axial\nprogenitor AP score")
# add r2
r2 = np.corrcoef(df["somite_count"], df["ap_score"])[0, 1] ** 2
plt.text(0.05, 0.95, f"R² = {r2:.2f}", transform=plt.gca().transAxes)
plt.xlim(0,34)
plt.xticks([0, 10, 20, 30])
plt.ylim(0.5, 0.85)
# add regression line
plt.show()
save_plot(plots_path / "axial_progenitor_ap_score_vs_somite_count.svg", fig)


## Sibling germ layer analysis

In [ ]:
nmp_progenitors.set_index("node", inplace=True)
for germ_layer in ["Ectoderm", "Mesoderm"]:
    nmp_progenitors[germ_layer] = tdata.obs.groupby("sibling_descendants")["germ_layer"].apply(lambda s: s.eq(germ_layer).any())
fig, ax = plt.subplots(figsize=(.73, 1.46), dpi=600)
res = bipotency_permutation_test(
    nmp_progenitors,
    lineages=("Ectoderm", "Mesoderm"),
    size_key = "n_siblings",
    ax=ax,
    bipotent_color = "#77649A"
)
save_plot(plots_path / "nmp_bipotent_barplot.svg", fig)

## Clonal dominance analysis

In [ ]:
py.tl.clades(tdata, depth = 7, depth_key = "time")
tdata.obs["contains_nmp"] = tdata.obs.groupby("clade")["cell_type"].transform(lambda x: "Axial progenitor" in x.values)
df = tdata.obs.drop_duplicates("clade").groupby("embryo").contains_nmp.mean().reset_index()
df["stage"] = df["embryo"].str.split("-").str[0]
df["nmp_percent"] = df["contains_nmp"] * 100

Plot

In [ ]:
fig, ax = plt.subplots(figsize=(1.71, 1.71), dpi=600)
sns.barplot(data = df.query("stage != 'E10.0'").query("stage != 'E7.5'"), x = "stage", y = "nmp_percent", 
            hue = "stage", palette = stage_palette, saturation = 1,capsize = 0.3, err_kws = {"linewidth": 0.8, "color": "black"})
sns.scatterplot(data = df.query("stage != 'E10.0'").query("stage != 'E7.5'"), x = "stage", y = "nmp_percent", 
                s = 20, legend = False, ax = ax, color= "black", edgecolor = None)
plt.ylabel("E6.0 clades containing\nAxial Progenitors (%)")
save_plot(plots_path / "clade_nmp_percent_barplot.svg", fig)

Diagram

In [ ]:
e8_example = cas.sim.birth_death_process(num_extant = 200)
cas.tl.rescale_node_times(e8_example, max = 8)
py.tl.clades(e8_example, depth = 6, depth_key = "time")
e9_example = cas.sim.birth_death_process(num_extant = 400)
cas.tl.rescale_node_times(e9_example, max = 9)
py.tl.clades(e9_example, depth = 6, depth_key = "time")
# randomly set 1% of values to True
e8_example.obs["contains_nmp"] = np.random.choice([True, False], size=e8_example.n_obs, p=[0.05, 0.95])
e8_example.obs["contains_nmp"] = e8_example.obs.groupby("clade")["contains_nmp"].transform("any")
e9_example.obs["contains_nmp"] = np.random.choice([True, False], size=e9_example.n_obs, p=[0.03, 0.97])
e9_example.obs["contains_nmp"] = e9_example.obs.groupby("clade")["contains_nmp"].transform("any")
fig, axes = plt.subplots(2,1,figsize=(1,2), dpi=600, sharex=True)
ax = axes[0]
py.pl.tree(e8_example, depth_key = "time", branch_color = "clade", ax = ax)
py.pl.annotation(e8_example, keys = "contains_nmp", ax = ax, width= 0.2,
                 palette = {True: celltype_palette["Axial progenitor"], False: "lightgray"})
plt.axvline(x=6, color='red', linestyle='--', linewidth=1)
ax = axes[1]
py.pl.tree(e9_example, depth_key = "time", branch_color = "clade", ax = ax)
py.pl.annotation(e9_example, keys = "contains_nmp", ax = ax, width= 0.18,
                 palette = {True: celltype_palette["Axial progenitor"], False: "lightgray"})
plt.axvline(x=6, color='red', linestyle='--', linewidth=1)
save_plot(plots_path / "axial_clade_example.svg", fig)

## Proliferation score

Define as number of neighbors within previous data (tree distance < 2)

In [ ]:
py.tl.tree_neighbors(tdata, depth_key = "time", max_dist=2, update = False)
tdata.obs["proliferation"] = (tdata.obsp["tree_distances"] > 0).sum(axis=1)

E9.5 by cell type

In [ ]:
df = tdata.obs.query("stage == 'E9.5'").groupby("cell_type")["proliferation"].mean().sort_values(ascending=False).reset_index()
df["rank"] = np.arange(len(df))
fig, ax = plt.subplots(figsize=(1.6, 1.6), dpi = 600)
sns.scatterplot(data=df, x="rank", y="proliferation", edgecolor ="black", color = "lightgray", ax = ax, s = 10)
plt.xticks([0,20,40,60,80]);
save_plot(plots_path / "proliferation_by_cell_type.svg", fig)

By timepoint and cell type

In [ ]:
tdata.obs["stage"] = pd.Categorical(tdata.obs["stage"], categories = ["E7.5","E8.0","E8.5","E9.0","E9.5","E10.0"], ordered=True)
df = tdata.obs.groupby(["cell_type","stage"])["proliferation"].mean().sort_values(ascending=False).unstack()
df = df.T
cmap = plt.cm.get_cmap("magma").copy()
cmap.set_bad(color="lightgray")
fig, ax = plt.subplots(figsize=(9,3), dpi=600, layout = "constrained")
sns.heatmap(df, xticklabels = df.columns, yticklabels=df.index, ax = ax,cmap = cmap)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=6, rotation = 90);
ax.set_yticklabels(ax.get_yticklabels(),fontsize=6);
save_plot(plots_path / "proliferation_heatmap.svg", fig, rasterize=True)

Diagram

In [ ]:
example = cas.sim.birth_death_process(num_extant = 50)
cas.tl.rescale_node_times(example, max = 1)
py.tl.tree_neighbors(example, depth_key = "time", max_dist=.4, update = False)
example.obs["proliferation"] = (example.obsp["tree_distances"] > 0).sum(axis=1)
fig, ax = plt.subplots(figsize=(1,2), dpi=600)
py.pl.tree(example, depth_key = "time", keys = "proliferation", ax = ax, annotation_width= 0.2, cmap = "magma")
## add dashed line at 0.2
plt.axvline(x=0.8, color='red', linestyle='--', linewidth=1)
save_plot(plots_path / "proliferation_example.svg", fig)